# Story 2: Can the Fed control inflation and maintain full employment?

**Author:** Muhammad Suffyan Khan  
**Course:** DATA 608 - Knowledge and Visual Analytics  

**Question.** Over the last 25 years, when did inflation and unemployment move closer to the Fed’s goals, and what happened around changes in the federal funds rate?

**Audience.** A general audience familiar with inflation and jobs, but not necessarily monetary policy. This is an observational account, not a causal estimate of the Fed's effect.

**Visualization tool.** Plotly. Story 1 used Matplotlib and Seaborn. This notebook attempts to obtain CPI and unemployment directly through the BLS public API using original Python code. If its anonymous daily request quota is exhausted, it explicitly reports a fallback to FRED's mirrors of those BLS series. It downloads the monthly federal funds series from FRED's published CSV endpoint. The FRED CSV endpoint is a programmatic data download, distinct from the keyed FRED API. Check the printed retrieval method before claiming API bonus credit.


In [11]:
from pathlib import Path
from datetime import date
import io, json, urllib.parse, urllib.request
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path.cwd()
if BASE.name == 'notebooks': BASE = BASE.parent
DATA = BASE / 'data' / 'processed'; FIG = BASE / 'figures'
DATA.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
START = pd.Timestamp('2001-09-01')
END = pd.Timestamp('2026-08-01') # September 2026 CPI and unemployment are not yet published

def get_bls(series_id):
    records = []
    # Public v2 API permits at most 10 years per request without a registration key.
    for start, end in [(2000, 2009), (2010, 2019), (2020, 2026)]:
        query = urllib.parse.urlencode({'startyear':start, 'endyear':end})
        url = f'https://api.bls.gov/publicAPI/v2/timeseries/data/{series_id}?{query}'
        with urllib.request.urlopen(url, timeout=35) as response:
            payload = json.load(response)
        if payload['status'] != 'REQUEST_SUCCEEDED':
            raise RuntimeError(f'BLS request failed: {payload.get("message")}')
        for row in payload['Results']['series'][0]['data']:
            if row['period'].startswith('M') and row['period'] != 'M13' and row['value'] not in ('-', ''):
                records.append((pd.Timestamp(int(row['year']),int(row['period'][1:]),1),float(row['value'])))
    s = pd.Series(dict(records), name=series_id).sort_index()
    if s.empty: raise ValueError(f'No BLS records for {series_id}')
    return s

# Seasonally adjusted CPI-U, all items, and seasonally adjusted civilian unemployment rate.
try:
    cpi = get_bls('CUSR0000SA0')
    unemployment = get_bls('LNS14000000')
    print('BLS public API retrieval succeeded for both series.')
except RuntimeError as exc:
    print(f'BLS public API unavailable ({exc}); using FRED mirrors of BLS series for this run.')
    def fred_bls(series):
        url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}'
        table = pd.read_csv(url,parse_dates=['observation_date']).set_index('observation_date')
        return pd.to_numeric(table[series],errors='coerce')
    cpi = fred_bls('CPIAUCSL')
    unemployment = fred_bls('UNRATE')
fred_url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=FEDFUNDS'
funds = pd.read_csv(fred_url, parse_dates=['observation_date']).set_index('observation_date')['FEDFUNDS']
funds = pd.to_numeric(funds, errors='coerce')

frame = pd.concat([cpi.rename('cpi_index'), unemployment.rename('unemployment'),
                   funds.rename('funds_rate')], axis=1).sort_index()
frame['inflation_yoy'] = (frame['cpi_index'] / frame['cpi_index'].shift(12) - 1) * 100
frame = frame.loc[START:END, ['cpi_index','inflation_yoy','funds_rate','unemployment']]
expected = pd.date_range(START,END,freq='MS')
frame = frame.reindex(expected)
missing = frame.index[frame[['inflation_yoy','funds_rate','unemployment']].isna().any(axis=1)]
if len(missing):
    print('Missing source observations, not interpolated:', ', '.join(x.strftime('%B %Y') for x in missing))
assert frame.index.is_unique and frame.index.is_monotonic_increasing
frame.index.name = 'month'
frame.to_csv(DATA / 'monthly_series.csv', float_format='%.4f')
print(f'{len(frame)} months: {frame.index.min():%B %Y}–{frame.index.max():%B %Y}')
display(frame.tail(12).round(2))


BLS public API unavailable (BLS request failed: ['Request could not be serviced, as the daily threshold for total number of requests allocated to the user with registration key  has been reached.']); using FRED mirrors of BLS series for this run.
Missing source observations, not interpolated: October 2025
300 months: September 2001–August 2026


,cpi_index,inflation_yoy,funds_rate,unemployment
month,,,,
2025-09-01,324.24,3.02,4.22,4.4
2025-10-01,NaN,NaN,4.09,NaN
2025-11-01,325.06,2.70,3.88,4.5
2025-12-01,326.03,2.65,3.72,4.4
2026-01-01,326.59,2.39,3.64,4.3
2026-02-01,327.46,2.43,3.64,4.4
2026-03-01,330.29,3.29,3.64,4.3
2026-04-01,332.41,3.78,3.64,4.3
2026-05-01,333.98,4.17,3.63,4.3


## Reading the evidence

Inflation is the **year-over-year percentage change** in the seasonally adjusted CPI-U index, not the CPI level. The federal funds rate is the monthly average effective rate. Unemployment is the seasonally adjusted monthly civilian unemployment rate. All three are percentages, but they measure different things; the charts separate their panels to avoid implying that identical numbers mean identical economic conditions.

The dashed 2% line is the Fed's longer-run **PCE** inflation goal used as a reference only. CPI inflation is not the PCE measure, so crossing that line cannot by itself determine whether the Fed met its formal goal. The Fed's employment objective is maximum employment, not a single numerical unemployment target. Missing published months are printed above and left as gaps rather than estimated.


In [12]:
COL = {'inflation_yoy':'#D55E00','funds_rate':'#3366AA','unemployment':'#157F70'}
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=.075,
                    subplot_titles=['Consumer price inflation (12-month CPI change)',
                                    'Effective federal funds rate', 'Unemployment rate'])
for i, (name,label) in enumerate([('inflation_yoy','Inflation'),('funds_rate','Funds rate'),('unemployment','Unemployment')],1):
    fig.add_trace(go.Scatter(x=frame.index,y=frame[name],mode='lines',connectgaps=False,name=label,
                             line=dict(color=COL[name],width=2.5),hovertemplate='%{x|%b %Y}: %{y:.1f}%<extra>'+label+'</extra>'),row=i,col=1)
    fig.update_yaxes(title_text='Percent',row=i,col=1)
fig.add_hline(y=2,line_color='#999999',line_dash='dash',row=1,col=1,
              )
fig.update_layout(title='Inflation, interest rates and unemployment over 25 years',height=880,
                  template='plotly_white',hovermode='x unified',legend=dict(orientation='h',y=-.07),
                  margin=dict(l=80,r=30,t=95,b=95))
fig.write_html(FIG/'01_twenty_five_year_timeline.html',include_plotlyjs=True)
fig.write_image(FIG/'01_twenty_five_year_timeline.png',scale=2)
fig.write_image(FIG/'01_twenty_five_year_timeline.pdf')
fig.show()


### Why this visual?
The aligned monthly timelines preserve the order of events and make the rate increases following inflation changes visible. Separate panels keep each measure readable, while the common time axis makes major episodes comparable. A line plot is more suitable than a single correlation because monetary policy and employment respond with delays, and both respond to other economic shocks.


In [13]:
zoom = frame.loc['2019-01':'2026-08']
fig2 = make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.12,
                     subplot_titles=['Inflation rose before the large rate increases',
                                     'Unemployment and the funds rate followed different paths'])
fig2.add_trace(go.Scatter(x=zoom.index,y=zoom.inflation_yoy,connectgaps=False,name='12-month CPI inflation',
                          line=dict(color=COL['inflation_yoy'],width=3)),row=1,col=1)
for name,label in [('funds_rate','Funds rate'),('unemployment','Unemployment')]:
    fig2.add_trace(go.Scatter(x=zoom.index,y=zoom[name],connectgaps=False,name=label,
                              line=dict(color=COL[name],width=2.5)),row=2,col=1)
for row in [1,2]: fig2.update_yaxes(title_text='Percent',row=row,col=1)
fig2.add_vrect(x0='2020-03-01',x1='2020-05-01',fillcolor='#aaaaaa',opacity=.15,line_width=0)
fig2.update_layout(title='The pandemic and inflation episode, 2019–2026',height=680,
                   template='plotly_white',hovermode='x unified',legend=dict(orientation='h',y=-.12),
                   margin=dict(l=75,r=30,t=90,b=100))
fig2.write_html(FIG/'02_pandemic_inflation_episode.html',include_plotlyjs=True)
fig2.write_image(FIG/'02_pandemic_inflation_episode.png',scale=2)
fig2.write_image(FIG/'02_pandemic_inflation_episode.pdf')
fig2.show()


In [14]:
for period,start,end in [('Pre-pandemic expansion','2015-01','2019-12'),
                                ('Pandemic and inflation','2020-01','2022-12'),
                                ('Disinflation period','2023-01','2026-08')]:
    x=frame.loc[start:end]
    print(f'{period}: inflation {x.inflation_yoy.iloc[0]:.1f}% to {x.inflation_yoy.iloc[-1]:.1f}%; '
          f'unemployment {x.unemployment.iloc[0]:.1f}% to {x.unemployment.iloc[-1]:.1f}%; '
          f'funds rate {x.funds_rate.iloc[0]:.2f}% to {x.funds_rate.iloc[-1]:.2f}%')
print('Inflation maximum:',frame.inflation_yoy.idxmax().strftime('%B %Y'),
      f'{frame.inflation_yoy.max():.1f}%')
print('Unemployment maximum:',frame.unemployment.idxmax().strftime('%B %Y'),
      f'{frame.unemployment.max():.1f}%')


Pre-pandemic expansion: inflation -0.2% to 2.3%; unemployment 5.7% to 3.6%; funds rate 0.11% to 1.55%
Pandemic and inflation: inflation 2.6% to 6.4%; unemployment 3.6% to 3.5%; funds rate 1.55% to 4.10%
Disinflation period: inflation 6.3% to 3.4%; unemployment 3.5% to 4.1%; funds rate 4.33% to 3.63%
Inflation maximum: June 2022 9.0%
Unemployment maximum: April 2020 14.8%


## Answer and limits

The Fed has **sometimes** seen low inflation and low unemployment at the same time, especially in the late 2010s. It has not maintained both outcomes continuously across 25 years. The 2008–09 crisis and 2020 pandemic sharply increased unemployment. CPI inflation peaked around **9.0% in June 2022** in these data, while the effective funds rate rose subsequently. Inflation eased to about **3.4% in August 2026** with unemployment about **4.1%**, though inflation rose again in spring 2026 after earlier declines. These outcomes show the objectives are not mechanically contradictory in every period, but neither has remained consistently satisfied. The rate series indicates policy response, but these descriptive comparisons cannot isolate the Fed's causal impact from supply shocks, fiscal policy, global events, and other factors. A 25-year chart cannot establish whether a particular level of unemployment equals maximum employment.

**Sources (retrieved when the notebook runs):** [BLS Public Data API](https://www.bls.gov/developers/), series [CUSR0000SA0](https://www.bls.gov/cpi/) and [LNS14000000](https://www.bls.gov/cps/); [FRED FEDFUNDS](https://fred.stlouisfed.org/series/FEDFUNDS); [Federal Reserve monetary policy goals](https://www.federalreserve.gov/monetarypolicy/files/FOMC_LongerRunGoals.pdf). Date range: September 2001 to August 2026. BLS series and FRED observations can be revised after retrieval.

